# From Loss to the First Parameter Update

> A configuration file can reconstruct a model's architecture, but newly initialized parameters have not learned any language patterns. When such a model sees `cats chase`, the correct answer is `mice`; instead, it may spread probability almost evenly across nine tokens or assign the highest probability to `sleep`.
>
> Merely telling the model that its guess was wrong is not enough. It needs a number that measures how wrong the guess was, and it must use that number to determine how every parameter should change. That number is the **loss**, and the direction of change comes from **backpropagation**.
>
> We will first teach a tiny model with a single parameter table to learn `cats chase mice`. After it genuinely changes from a wrong prediction to a correct one, we will add batches, padding, and the **label mask** used in SFT. Each step leaves probabilities, curves, or assertions as evidence.

The sentence `cats chase mice` can be split into consecutive exercises: see `<bos>` and predict `cats`; see `cats` and predict `chase`; see `chase` and predict `mice`.

A newly initialized model does not know these relationships. Training therefore needs a number that measures the gap between its prediction and the correct answer, then sends that gap back to the parameters.

The first number is called **loss**. The process of calculating parameter-update directions backward through the computation is called **backpropagation**. We will use one small parameter table to observe a complete prediction and its first update.


## 0. The Training Objective of a Language Model

Start with a sentence containing only three ordinary words:

```text
cats chase mice
```

After adding `<bos>` and `<eos>`, the model receives four consecutive exercises:

| Last token already seen | Next token to predict |
|:---|:---|
| `<bos>` | `cats` |
| `cats` | `chase` |
| `chase` | `mice` |
| `mice` | `<eos>` |

**Next-token prediction** means predicting the next token from the tokens that have already appeared. In plain language, it turns one sentence into many small “what comes next?” questions.

To make every training step visible, we begin with a Bigram Language Model. It predicts the next token using only the current token, without reading earlier context. The model is small, but its loss, backpropagation, and parameter updates follow the same rules as larger language models.


In [ ]:
import math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, Dataset

torch.manual_seed(42)

tokens = [
    "<pad>", "<bos>", "<eos>", "cats", "dogs",
    "chase", "mice", "balls", "sleep",
]
token_to_id = {token: index for index, token in enumerate(tokens)}

sentence_tokens = ["<bos>", "cats", "chase", "mice", "<eos>"]
sentence = torch.tensor([token_to_id[token] for token in sentence_tokens])
input_ids = sentence[:-1]
labels = sentence[1:]

target_map = torch.zeros(len(tokens), len(tokens))
target_map[input_ids, labels] = 1

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(target_map, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(tokens)), tokens, rotation=45, ha="right")
ax.set_yticks(range(len(tokens)), tokens)
ax.set_xlabel("Target token")
ax.set_ylabel("Current token")
ax.set_title("Four next-token training pairs")
plt.tight_layout()
plt.show()

assert torch.equal(input_ids, torch.tensor([1, 3, 5, 6]))
assert torch.equal(labels, torch.tensor([3, 5, 6, 2]))


Each blue square in the figure is one training example. The horizontal axis is the answer, and the vertical axis is the current token.

The model also maintains a $9\times9$ parameter table. Given a token ID, it selects the corresponding row and obtains raw scores for the nine candidate tokens.

These raw scores are called **logits**. Logits are not probabilities and may be any real numbers. Softmax converts them into probabilities that sum to 1.


In [ ]:
class BigramLanguageModel(nn.Module):
    """A tiny language model that predicts the next token from only the current token."""

    def __init__(self, vocab_size):
        super().__init__()
        self.transition = nn.Embedding(vocab_size, vocab_size)

    def forward(self, input_ids, attention_mask=None, labels=None):
        """Return logits and, when labels are supplied, the mean Cross-Entropy Loss."""
        logits = self.transition(input_ids)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
            )
        return {"logits": logits, "loss": loss}


model = BigramLanguageModel(vocab_size=len(tokens))
with torch.no_grad():
    initial_logits = model(input_ids.unsqueeze(0))["logits"][0]
    initial_probs = F.softmax(initial_logits, dim=-1)

fig, ax = plt.subplots(figsize=(8, 3.8))
image = ax.imshow(initial_probs, cmap="YlOrRd", vmin=0, vmax=0.4)
ax.set_xticks(range(len(tokens)), tokens, rotation=45, ha="right")
ax.set_yticks(range(len(input_ids)), sentence_tokens[:-1])
ax.set_xlabel("Predicted next token")
ax.set_ylabel("Current token")
ax.set_title("Probabilities before training")
fig.colorbar(image, ax=ax, label="Probability")
plt.tight_layout()
plt.show()

assert initial_probs.shape == (4, 9)
assert torch.allclose(initial_probs.sum(dim=-1), torch.ones(4))


At initialization, the colors in each row are scattered. The correct answer is not necessarily the darkest square because the parameter table is still random.

We now need a single score with two properties: a lower probability for the correct token should produce a larger score, while a probability near 1 should produce a score near 0. Cross-Entropy Loss provides exactly this behavior.


## 1. Cross-Entropy at One Position

Consider one prediction. Suppose the vocabulary has four candidate tokens and the model produces these logits:

$$[2.0,\ 1.0,\ 0.1,\ -1.0]$$

The correct answer is token 0. The calculation has two steps.

First, softmax converts logits into probabilities:

$$p_i=\frac{e^{z_i}}{\sum_j e^{z_j}}$$

Second, select the probability assigned to the correct token and take its negative logarithm:

$$L=-\log p_{correct}$$

This is Cross-Entropy Loss for classification. In a language model, every position is a classification problem that selects the next token from the full vocabulary.


In [ ]:
demo_logits = torch.tensor([2.0, 1.0, 0.1, -1.0])
correct_id = 0

exp_values = torch.exp(demo_logits)
demo_probs = exp_values / exp_values.sum()
correct_prob = demo_probs[correct_id].item()
manual_loss = -math.log(correct_prob)
torch_loss = F.cross_entropy(
    demo_logits.unsqueeze(0),
    torch.tensor([correct_id]),
)

colors = ["tab:blue", "lightgray", "lightgray", "lightgray"]
fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(["A", "B", "C", "D"], demo_probs, color=colors)
ax.bar_label(bars, fmt="%.3f")
ax.set_ylim(0, 1)
ax.set_ylabel("Probability")
ax.set_title(f"Correct probability = {correct_prob:.3f}, loss = {manual_loss:.3f}")
plt.tight_layout()
plt.show()

assert abs(manual_loss - torch_loss.item()) < 1e-6


The hand calculation matches PyTorch. Cross-Entropy does not care what the incorrect tokens are called; it checks how much probability the model assigned to the correct token.

Why take the negative logarithm instead of using $1-p_{correct}$ directly? The curve below makes the difference easier to see.


In [ ]:
probability = torch.linspace(0.01, 0.99, 200)
loss_curve = -torch.log(probability)
marked_probs = torch.tensor([0.1, 0.5, 0.9])
marked_losses = -torch.log(marked_probs)

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(probability, loss_curve, color="tab:blue")
ax.scatter(marked_probs, marked_losses, color="tab:red", zorder=3)
for prob, loss_value in zip(marked_probs, marked_losses):
    ax.annotate(
        f"p={prob:.1f}, loss={loss_value:.2f}",
        (prob, loss_value),
        xytext=(8, 8),
        textcoords="offset points",
    )
ax.set_xlabel("Probability of the correct token")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Low confidence receives a larger penalty")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


When the correct-token probability falls from 0.9 to 0.5, loss rises from 0.11 to 0.69. If it falls to 0.1, loss rises to 2.30. A confidently wrong model receives a larger penalty.

So far, loss only measures the error. Training must answer a second question: how should the logits change so that the next loss becomes smaller?


## 2. One Parameter Update

Treat the four logits above as trainable parameters. Backpropagation calculates how each parameter affects the loss. This effect is called a gradient.

**Gradient** is the sensitivity of loss to a change in a parameter. In plain language, it tells the optimizer which direction will reduce the loss.

We will perform one SGD update and compare the probabilities before and after it.


In [ ]:
trainable_logits = nn.Parameter(torch.tensor([2.0, 1.0, 0.1, -1.0]))
optimizer = torch.optim.SGD([trainable_logits], lr=0.5)
target = torch.tensor([correct_id])

before_probs = F.softmax(trainable_logits.detach(), dim=-1)
before_loss = F.cross_entropy(trainable_logits.unsqueeze(0), target)

optimizer.zero_grad()
before_loss.backward()
gradient = trainable_logits.grad.detach().clone()
optimizer.step()

after_probs = F.softmax(trainable_logits.detach(), dim=-1)
after_loss = F.cross_entropy(trainable_logits.unsqueeze(0), target)

x = torch.arange(4)
width = 0.36
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.bar(x - width / 2, before_probs, width, label="Before")
ax.bar(x + width / 2, after_probs, width, label="After")
ax.set_xticks(x, ["A", "B", "C", "D"])
ax.set_ylim(0, 1)
ax.set_ylabel("Probability")
ax.set_title("One gradient step raises the correct probability")
ax.legend()
plt.tight_layout()
plt.show()

assert gradient[correct_id] < 0
assert after_probs[correct_id] > before_probs[correct_id]
assert after_loss < before_loss


The gradient for the correct token is negative. SGD updates a parameter with $\theta\leftarrow\theta-\eta g$, so subtracting a negative gradient raises the correct token's logit. The probabilities of the other tokens fall accordingly.

This step contains the core causal chain of training: loss produces gradients, gradients change parameters, and changed parameters alter the next prediction. We can now extend the same process to a full sentence.


## 3. Sequence-Level Training

A sentence has four prediction positions, so it produces four losses. By default, PyTorch averages the losses over all valid positions:

$$L_{sentence}=\frac{L_1+L_2+L_3+L_4}{4}$$

First, inspect which questions are hardest before training.


In [ ]:
sentence_model = BigramLanguageModel(vocab_size=len(tokens))
sentence_logits = sentence_model(input_ids.unsqueeze(0))["logits"][0]
position_losses = F.cross_entropy(
    sentence_logits,
    labels,
    reduction="none",
)
mean_loss = F.cross_entropy(sentence_logits, labels)

pair_names = [
    f"{current}→{target}"
    for current, target in zip(sentence_tokens[:-1], sentence_tokens[1:])
]
fig, ax = plt.subplots(figsize=(7, 3.8))
bars = ax.bar(pair_names, position_losses.detach(), color="tab:orange")
ax.axhline(mean_loss.item(), color="tab:blue", linestyle="--", label="Mean")
ax.bar_label(bars, fmt="%.2f")
ax.set_ylabel("Loss")
ax.set_title("Loss at each position before training")
ax.tick_params(axis="x", rotation=25)
ax.legend()
plt.tight_layout()
plt.show()

assert torch.allclose(position_losses.mean(), mean_loss)


The positions have different difficulty, but one `backward()` call calculates gradients of the mean loss for every relevant parameter. We will repeat the update 80 times and record two quantities: mean loss and mean probability assigned to the correct tokens.


In [ ]:
optimizer = torch.optim.AdamW(sentence_model.parameters(), lr=0.12)
loss_history = []
correct_prob_history = []

for step in range(80):
    outputs = sentence_model(
        input_ids.unsqueeze(0),
        labels=labels.unsqueeze(0),
    )
    loss = outputs["loss"]

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        probs = F.softmax(outputs["logits"][0], dim=-1)
        correct_probs = probs[torch.arange(len(labels)), labels]
        loss_history.append(loss.item())
        correct_prob_history.append(correct_probs.mean().item())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(loss_history, color="tab:blue")
axes[0].set_title("Training loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.25)

axes[1].plot(correct_prob_history, color="tab:green")
axes[1].set_title("Mean probability of correct tokens")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Probability")
axes[1].set_ylim(0, 1.05)
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

assert loss_history[-1] < loss_history[0] * 0.1
assert correct_prob_history[-1] > 0.9


In [ ]:
with torch.no_grad():
    trained_logits = sentence_model(input_ids.unsqueeze(0))["logits"][0]
    predicted_ids = trained_logits.argmax(dim=-1)

predicted_tokens = [tokens[index] for index in predicted_ids.tolist()]
target_tokens = sentence_tokens[1:]

fig, ax = plt.subplots(figsize=(8, 2.2))
ax.axis("off")
table = ax.table(
    cellText=[sentence_tokens[:-1], target_tokens, predicted_tokens],
    rowLabels=["Current", "Target", "Prediction"],
    colLabels=[f"Position {index}" for index in range(4)],
    cellLoc="center",
    loc="center",
)
table.scale(1, 1.6)
ax.set_title("Predictions after training", pad=16)
plt.tight_layout()
plt.show()

assert predicted_tokens == target_tokens


The model now answers all four questions correctly. No separate human labels were added: `labels` is simply the original sentence shifted left by one position, so each input token is paired with its next token.

This result only shows that the model memorized one sentence. Real training processes many samples of different lengths, which introduces the next practical problem.


## 4. Batches of Variable-Length Sequences

Add a second sentence:

```text
cats chase mice   -> 4 prediction positions
cats sleep        -> 3 prediction positions
```

Every row in a two-dimensional tensor must have the same length, so the shorter sentence is extended with `<pad>`. Padding creates two new requirements: the model should not attend to PAD positions, and the loss should not treat PAD as an answer.

**Data Collator** is a component that assembles individual samples into a batch. In plain language, it pads every sample to the longest sequence in the current batch and creates the corresponding masks.


In [ ]:
def simple_collate(features, pad_id=0, ignore_index=-100):
    """Pad samples of different lengths into one batch."""
    max_len = max(len(item["input_ids"]) for item in features)
    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []

    for item in features:
        pad_len = max_len - len(item["input_ids"])
        batch_input_ids.append(item["input_ids"] + [pad_id] * pad_len)
        batch_attention_mask.append(
            [1] * len(item["input_ids"]) + [0] * pad_len
        )
        batch_labels.append(
            item["labels"] + [ignore_index] * pad_len
        )

    return {
        "input_ids": torch.tensor(batch_input_ids),
        "attention_mask": torch.tensor(batch_attention_mask),
        "labels": torch.tensor(batch_labels),
    }


short_sentence = ["<bos>", "cats", "sleep", "<eos>"]
short_ids = [token_to_id[token] for token in short_sentence]
features = [
    {"input_ids": sentence[:-1].tolist(), "labels": sentence[1:].tolist()},
    {"input_ids": short_ids[:-1], "labels": short_ids[1:]},
]
batch = simple_collate(features)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
items = [
    ("input_ids", "Input IDs", "Blues"),
    ("attention_mask", "Attention mask", "Greens"),
    ("labels", "Labels", "Oranges"),
]
for ax, (name, title, color_map) in zip(axes, items):
    values = batch[name]
    ax.imshow(values, cmap=color_map, aspect="auto")
    for row in range(values.size(0)):
        for column in range(values.size(1)):
            ax.text(column, row, str(values[row, column].item()), ha="center")
    ax.set_xticks(range(values.size(1)))
    ax.set_yticks([0, 1], ["Long", "Short"])
    ax.set_xlabel("Position")
    ax.set_title(title)
plt.tight_layout()
plt.show()

assert batch["input_ids"].shape == (2, 4)
assert batch["attention_mask"][1, -1].item() == 0
assert batch["labels"][1, -1].item() == -100


The three tensors have different jobs:

- `input_ids` uses the `<pad>` ID to make shapes equal.
- `attention_mask=0` tells the model that a position is PAD and should not participate in attention.
- `labels=-100` tells Cross-Entropy that a position should not contribute to loss.

The last two settings are not interchangeable. `attention_mask` controls which positions the model may read; `labels` controls which predictions receive a score.


## 5. Label Masks in SFT

In pretraining text, nearly every ordinary token can serve as a prediction target. SFT (Supervised Fine-Tuning) has a different goal for conversations: we primarily want the model to learn the assistant's response.

**Chat Template** is a template that converts structured messages with roles such as `user` and `assistant` into a token sequence. In plain language, it adds role markers that the model recognizes.

Consider a minimal templated example:

```text
Full sequence: <user> 2+2? <assistant> 4 <eos>
Input:         <user> 2+2? <assistant> 4
Target:        ignore ignore 4           <eos>
```

Only the assistant response `4 <eos>` contributes to loss. The user question and role markers use `-100` and are ignored.


In [ ]:
chat_tokens = ["<user>", "2+2?", "<assistant>", "4", "<eos>"]
chat_vocab = {token: index for index, token in enumerate(chat_tokens)}
chat_input_tokens = chat_tokens[:-1]
chat_labels = torch.tensor([-100, -100, chat_vocab["4"], chat_vocab["<eos>"]])
loss_mask = (chat_labels != -100).float().unsqueeze(0)

fig, ax = plt.subplots(figsize=(7, 1.8))
ax.imshow(loss_mask, cmap="Greens", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(chat_input_tokens)), chat_input_tokens)
ax.set_yticks([])
ax.set_xlabel("Current token position")
ax.set_title("Green positions contribute to SFT loss")
for column, included in enumerate(loss_mask[0]):
    label = "include" if included.item() == 1 else "ignore"
    ax.text(column, 0, label, ha="center", va="center")
plt.tight_layout()
plt.show()

assert chat_labels.tolist() == [-100, -100, 3, 4]


In [ ]:
torch.manual_seed(7)
chat_logits = torch.randn(len(chat_input_tokens), len(chat_vocab))
valid_positions = chat_labels != -100
valid_losses = F.cross_entropy(
    chat_logits[valid_positions],
    chat_labels[valid_positions],
    reduction="none",
)
display_losses = torch.zeros(len(chat_input_tokens))
display_losses[valid_positions] = valid_losses.detach()
sft_loss = valid_losses.mean()

colors = ["lightgray" if not valid else "tab:green" for valid in valid_positions]
fig, ax = plt.subplots(figsize=(7, 3.4))
bars = ax.bar(chat_input_tokens, display_losses, color=colors)
for bar, valid, value in zip(bars, valid_positions, display_losses):
    label = f"{value.item():.2f}" if valid else "ignored"
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), label, ha="center")
ax.set_ylabel("Token loss")
ax.set_title(f"SFT averages only green positions: loss = {sft_loss:.2f}")
plt.tight_layout()
plt.show()

assert valid_positions.sum().item() == 2
assert torch.allclose(sft_loss, valid_losses.mean())


The gray positions still pass through the model and produce logits, but they do not enter the final mean loss. Gradients therefore come mainly from the assistant's answer.

Whether to mask user and system content depends on the training objective and recipe. This example shows common assistant-only SFT. A recipe that also teaches the full conversation format may include more positions in the loss.


## 6. A Complete Training Loop

We have separately verified sentence loss, a parameter update, and batching. Now train one model on two kinds of sentences:

```text
cats chase mice
dogs chase balls
```

The Dataset returns one sample at a time. The DataLoader and collator assemble samples into batches. Each code cell still leaves one result that can be checked.


In [ ]:
class ToyTextDataset(Dataset):
    """Convert complete token sequences into input_ids and labels shifted one position right."""

    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        ids = self.sequences[index]
        return {"input_ids": ids[:-1], "labels": ids[1:]}


cats_sequence = ["<bos>", "cats", "chase", "mice", "<eos>"]
dogs_sequence = ["<bos>", "dogs", "chase", "balls", "<eos>"]
training_sequences = []
for sequence_tokens in [cats_sequence, dogs_sequence] * 4:
    training_sequences.append([token_to_id[token] for token in sequence_tokens])

dataset = ToyTextDataset(training_sequences)
sequence_lengths = [len(item) - 1 for item in training_sequences]

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.bar(range(len(sequence_lengths)), sequence_lengths, color="tab:purple")
ax.set_xticks(range(len(sequence_lengths)))
ax.set_xlabel("Sample index")
ax.set_ylabel("Prediction positions")
ax.set_title("Eight training samples")
ax.set_ylim(0, 5)
plt.tight_layout()
plt.show()

assert len(dataset) == 8


In [ ]:
torch.manual_seed(42)
dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=simple_collate,
)
batch_model = BigramLanguageModel(vocab_size=len(tokens))
batch_optimizer = torch.optim.AdamW(batch_model.parameters(), lr=0.08)
total_steps = 30 * len(dataloader)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    batch_optimizer,
    lr_lambda=lambda step: max(0.1, 1 - step / total_steps),
)

step_losses = []
learning_rates = []
gradient_norms = []

for epoch in range(30):
    for batch in dataloader:
        outputs = batch_model(**batch)
        loss = outputs["loss"]

        batch_optimizer.zero_grad()
        loss.backward()
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            batch_model.parameters(),
            max_norm=1.0,
        )
        batch_optimizer.step()
        scheduler.step()

        step_losses.append(loss.item())
        gradient_norms.append(gradient_norm.item())
        learning_rates.append(scheduler.get_last_lr()[0])

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(step_losses, color="tab:blue")
axes[0].set_title("Batch loss")
axes[0].set_xlabel("Optimizer step")
axes[0].set_ylabel("Loss")

axes[1].plot(gradient_norms, color="tab:orange")
axes[1].set_title("Gradient norm")
axes[1].set_xlabel("Optimizer step")
axes[1].set_ylabel("Norm")

axes[2].plot(learning_rates, color="tab:green")
axes[2].set_title("Learning rate")
axes[2].set_xlabel("Optimizer step")
axes[2].set_ylabel("LR")
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

assert step_losses[-1] < step_losses[0]
assert learning_rates[-1] < learning_rates[0]


Every line in this training loop has an observable effect:

- `model(**batch)` produces logits and loss.
- `loss.backward()` produces gradients; the middle curve on the right records their norm.
- `optimizer.step()` updates parameters from those gradients, causing the loss curve on the left to fall.
- `scheduler.step()` changes the learning rate used by later updates; the right-hand plot shows this change.

A Trainer repeatedly performs these core steps reliably, then adds gradient accumulation, mixed precision, evaluation, and checkpoints around them. After this experiment, each layer hidden inside `trainer.train()` corresponds to an operation we have already run.


## 7. Checking the Training Result

A falling training loss does not prove that the model understands language. Our Bigram model sees only the current token. The training data contains two questions:

```text
cats chase -> mice
dogs chase -> balls
```

Both targets follow `chase`, but the Bigram model cannot see the earlier `cats` or `dogs`. It can only learn that half of the time `chase` is followed by `mice` and half by `balls`. We will inspect this stable failure directly.


In [ ]:
chase_id = token_to_id["chase"]
with torch.no_grad():
    chase_logits = batch_model(torch.tensor([chase_id]))["logits"][0]
    chase_probs = F.softmax(chase_logits, dim=-1)

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = [
    "tab:orange" if token in {"mice", "balls"} else "lightgray"
    for token in tokens
]
bars = ax.bar(tokens, chase_probs, color=colors)
ax.bar_label(bars, fmt="%.2f")
ax.set_ylim(0, 1)
ax.set_ylabel("Probability")
ax.set_title("Bigram model cannot use earlier context")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

mice_prob = chase_probs[token_to_id["mice"]].item()
balls_prob = chase_probs[token_to_id["balls"]].item()
assert abs(mice_prob - balls_prob) < 0.12
assert mice_prob + balls_prob > 0.85


This failure does not mean that the optimizer failed. The model's input lacks the information needed to solve the task. With a longer context, a Transformer can let `cats` or `dogs` influence the prediction after `chase`.

When reading training curves, distinguish at least two questions: is the loss decreasing, and can the model architecture represent the pattern required by the task? The first checks the training process; the second checks model capacity.


## 8. Mapping the Pieces to Industrial Training Libraries

The correspondence to Hugging Face Transformers or ModelScope `ms-swift` is now straightforward:

| Object used in this chapter | Common object in a training library |
|:---|:---|
| `ToyTextDataset` | `Dataset` or another training-dataset wrapper |
| `simple_collate` | Data Collator |
| `BigramLanguageModel` | A model such as `AutoModelForCausalLM` |
| `loss.backward()` | The Trainer's internal backward pass |
| `AdamW.step()` | An optimizer created or accepted by the Trainer |
| `scheduler.step()` | Learning-rate scheduler |
| Loss and gradient-norm curves | Logging and monitoring |

Industrial libraries change scale and engineering details, not the causal chain verified here: a lower correct-token probability produces a larger loss; loss changes parameters through gradients; and the new parameters change later probabilities.


## Summary

Check that you can answer each question independently:

- [ ] I can split one sentence into multiple next-token prediction pairs.
- [ ] I can calculate softmax, the correct-token probability, and Cross-Entropy Loss from logits by hand.
- [ ] I know how one `backward()` and `optimizer.step()` increase the correct token's probability.
- [ ] I can explain why sentence loss is the mean over all valid token losses.
- [ ] I can distinguish the roles of `attention_mask=0` and `labels=-100`.
- [ ] I know why assistant-only SFT masks user and template positions.
- [ ] I can inspect loss, gradient-norm, and learning-rate curves to check a training loop.
- [ ] I know why decreasing loss does not prove that the model architecture can solve the task.


## Exercises

> You may ask AI for hints, help breaking down the steps, or a direction check, but avoid asking it to complete the entire exercise.

### Exercise 1: Calculate Loss from the Correct Probability

At one position, the model assigns probability 0.25 to the correct token. Calculate the Cross-Entropy Loss at this position.

Hint: use $-\log(p)$. In Python, call `math.log`.


In [ ]:
correct_probability = 0.25

# TODO: replace the triple-quoted text with the loss calculation
exercise_loss = """Calculate -log(correct_probability) here"""

assert not isinstance(exercise_loss, str), "Replace the placeholder first"
assert abs(exercise_loss - 1.386294) < 1e-5, exercise_loss
print("Exercise 1 passed: you can convert the correct token's probability into Cross-Entropy Loss.")


### Exercise 2: Pad the Labels for a Short Sentence

A short sentence has three valid labels, while the longest sequence in the batch has five positions. Complete the full `labels` sequence.

Hint: PAD positions do not participate in loss, so use `-100`.


In [ ]:
valid_labels = [3, 5, 2]
max_length = 5

# TODO: Replace the triple-quoted content below with your code
padded_labels = 'TODO: replace this placeholder with your code'

assert not isinstance(padded_labels, str), 'Please replace the placeholder before running the assertion.'
assert padded_labels == [3, 5, 2, -100, -100], padded_labels
print("Exercise 2 passed: you can exclude PAD positions from loss with -100.")


### Exercise 3: Complete One Parameter Update

The code below prepares a trainable parameter containing three logits. Add backpropagation and an optimizer update so that the probability of the correct token increases.

Hint: call `loss.backward()` and then `optimizer.step()`.


In [ ]:
exercise_logits = nn.Parameter(torch.tensor([0.2, 0.1, -0.3]))
exercise_optimizer = torch.optim.SGD([exercise_logits], lr=0.5)
exercise_target = torch.tensor([1])
before = F.softmax(exercise_logits.detach(), dim=-1)[1].item()
exercise_loss = F.cross_entropy(exercise_logits.unsqueeze(0), exercise_target)

exercise_optimizer.zero_grad()
# TODO: add two lines here: backpropagate, then update the parameters

after = F.softmax(exercise_logits.detach(), dim=-1)[1].item()
assert after > before, f"The correct token's probability did not increase: {before:.3f} -> {after:.3f}"
print("Exercise 3 passed: you independently completed one loss -> gradient -> update cycle.")


## References

- [PyTorch CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) — formal definitions of logits, targets, and `ignore_index`
- [PyTorch autograd](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html) — computation graphs, gradients, and `backward()`
- [PyTorch AdamW](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html) — the optimizer interface used by this chapter's training loop
- [Hugging Face causal language modeling](https://huggingface.co/docs/transformers/tasks/language_modeling) — data preparation and training entry points for a real Causal LM
